# Keyword Price Outlier Check

크롤링 결과 CSV를 기준으로 `keyword`별 가격 분포를 확인하고, IQR 방식으로 이상치 후보를 추출합니다.

- 입력: `crawling/results`의 최신 `통합조회_전체_no_filter_*.csv`
- 기준: keyword별 Q1, Q3, IQR
- 이상치: `price < Q1 - 1.5 * IQR` 또는 `price > Q3 + 1.5 * IQR`
- 출력: `analysis/results/price_outliers`


In [15]:
from pathlib import Path

import pandas as pd


def resolve_analysis_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "analysis":
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "handoff").exists():
        return cwd.parent

    candidate = cwd / "code/backend/src/main/python/analysis"
    if candidate.exists():
        return candidate

    nested = cwd / "analysis"
    if nested.exists() and (nested / "handoff").exists():
        return nested

    return cwd

ANALYSIS_DIR = resolve_analysis_dir()
PYTHON_DIR = ANALYSIS_DIR.parent
CRAWLING_RESULTS_DIR = PYTHON_DIR / "crawling" / "results"
OUTPUT_DIR = ANALYSIS_DIR / "results" / "price_outliers"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidates = sorted(
    CRAWLING_RESULTS_DIR.glob("통합조회_전체_no_filter_*.csv"),
    key=lambda path: path.stat().st_mtime,
)

if not candidates:
    raise FileNotFoundError(f"no-filter 크롤링 CSV를 찾을 수 없습니다: {CRAWLING_RESULTS_DIR}")

csv_path = candidates[-1]
csv_path


WindowsPath('C:/project/kdtproject/kdtproject/code/backend/src/main/python/crawling/results/통합조회_전체_no_filter_20260522_1106.csv')

In [16]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")

working_df = df.copy()
working_df["price_numeric"] = pd.to_numeric(working_df["price"], errors="coerce")
working_df = working_df.dropna(subset=["keyword", "name", "price_numeric"])
working_df = working_df[working_df["price_numeric"] > 0].copy()
working_df["price_numeric"] = working_df["price_numeric"].astype(int)

print(f"source: {csv_path}")
print(f"rows: {len(df):,} -> valid price rows: {len(working_df):,}")
working_df.head()


source: C:\project\kdtproject\kdtproject\code\backend\src\main\python\crawling\results\통합조회_전체_no_filter_20260522_1106.csv
rows: 24,989 -> valid price rows: 24,988


,keyword,platform,pid,name,price,status,image_url,link,date,matched_keywords,canonical_name,price_numeric
0,게이밍 노트북,번개장터,408765604,삼성 노트북 게이밍 i5 GTX1650 477GB 판매,450000,판매중,https://media.bunjang.co.kr/product/408765604_...,https://m.bunjang.co.kr/products/408765604,2026-05-22 11:20,게이밍 노트북,게이밍 노트북,450000
1,게이밍 노트북,번개장터,409446999,SPARQ 노트북 본체 GEFORCE 게이밍 노트북 부품용,89000,판매중,https://media.bunjang.co.kr/product/409446999_...,https://m.bunjang.co.kr/products/409446999,2026-05-22 11:19,게이밍 노트북,게이밍 노트북,89000
2,게이밍 노트북,번개장터,409248995,삼성 노트북 화이트 Core i5 게이밍,129000,판매중,https://media.bunjang.co.kr/product/409248995_...,https://m.bunjang.co.kr/products/409248995,2026-05-22 11:19,게이밍 노트북,게이밍 노트북,129000
3,게이밍 노트북,번개장터,409524108,[미개봉]HP 15인치 게이밍노트북 오멘(240/5060/램24/1TB),1580000,판매중,https://media.bunjang.co.kr/product/409524108_...,https://m.bunjang.co.kr/products/409524108,2026-05-22 11:17,게이밍 노트북,게이밍 노트북,1580000
4,게이밍 노트북,번개장터,408052382,[미개봉]에이서 프레데터 헬리오스 네오 16 AI 5070 게이밍노트북,2250000,판매중,https://media.bunjang.co.kr/product/408052382_...,https://m.bunjang.co.kr/products/408052382,2026-05-22 10:58,게이밍 노트북,게이밍 노트북,2250000


In [17]:
LOW_PRICE_MEDIAN_RATIO = 0.2

keyword_stats_df = (
    working_df.groupby("keyword")["price_numeric"]
    .agg(
        item_count="count",
        min_price="min",
        q1=lambda values: values.quantile(0.25),
        median_price="median",
        q3=lambda values: values.quantile(0.75),
        max_price="max",
        average_price="mean",
    )
    .reset_index()
)

keyword_stats_df["iqr"] = keyword_stats_df["q3"] - keyword_stats_df["q1"]
keyword_stats_df["iqr_lower_bound"] = (keyword_stats_df["q1"] - 1.5 * keyword_stats_df["iqr"]).clip(lower=0)
keyword_stats_df["median_ratio_lower_bound"] = keyword_stats_df["median_price"] * LOW_PRICE_MEDIAN_RATIO
keyword_stats_df["lower_bound"] = keyword_stats_df[["iqr_lower_bound", "median_ratio_lower_bound"]].max(axis=1)
keyword_stats_df["upper_bound"] = keyword_stats_df["q3"] + 1.5 * keyword_stats_df["iqr"]

keyword_stats_df = keyword_stats_df.sort_values(["item_count", "keyword"], ascending=[False, True])
keyword_stats_df.head(30)


,keyword,item_count,min_price,q1,median_price,q3,max_price,average_price,iqr,iqr_lower_bound,median_ratio_lower_bound,lower_bound,upper_bound
4,갤럭시,10414,500,82250.00,159000.0,330000.0,999999999,3.834173e+05,247750.00,0.0,31800.0,31800.0,7.016250e+05
14,세이코,1986,168,140000.00,250000.0,520000.0,12250000,6.385428e+05,380000.00,0.0,50000.0,50000.0,1.090000e+06
16,스텔라이브,1868,1,10000.00,28000.0,70000.0,999999999,5.177838e+06,60000.00,0.0,5600.0,5600.0,1.600000e+05
19,핫토이,1812,100,111777.75,220000.0,350000.0,999999999,1.083885e+06,238222.25,0.0,44000.0,44000.0,7.073334e+05
7,던스,1243,1,35000.00,60000.0,105000.0,4444444,9.060930e+04,70000.00,0.0,12000.0,12000.0,2.100000e+05
2,ps5,1018,1,23625.00,45000.0,120000.0,90105000,2.597428e+05,96375.00,0.0,9000.0,9000.0,2.645625e+05
8,만년필,996,100,45000.00,150000.0,450000.0,99999999,9.311718e+05,405000.00,0.0,30000.0,30000.0,1.057500e+06
12,바이씨니,951,10000,50000.00,70000.0,105000.0,999999,8.685220e+04,55000.00,0.0,14000.0,14000.0,1.875000e+05
5,게이밍 노트북,885,500,450000.00,950000.0,1530000.0,9500000,1.210199e+06,1080000.00,0.0,190000.0,190000.0,3.150000e+06
18,제라늄,799,1111,8000.00,12345.0,25000.0,420000,2.114209e+04,17000.00,0.0,2469.0,2469.0,5.050000e+04


In [18]:
outlier_df = working_df.merge(
    keyword_stats_df[[
        "keyword",
        "q1",
        "q3",
        "iqr",
        "iqr_lower_bound",
        "median_ratio_lower_bound",
        "lower_bound",
        "upper_bound",
        "median_price",
    ]],
    on="keyword",
    how="left",
)

outlier_df["outlier_type"] = ""
outlier_df.loc[outlier_df["price_numeric"] < outlier_df["lower_bound"], "outlier_type"] = "low"
outlier_df.loc[outlier_df["price_numeric"] > outlier_df["upper_bound"], "outlier_type"] = "high"

outlier_df = outlier_df[outlier_df["outlier_type"] != ""].copy()
outlier_df["price_to_median_ratio"] = outlier_df["price_numeric"] / outlier_df["median_price"]

outlier_columns = [
    "keyword",
    "platform",
    "pid",
    "name",
    "price_numeric",
    "outlier_type",
    "lower_bound",
    "iqr_lower_bound",
    "median_ratio_lower_bound",
    "q1",
    "median_price",
    "q3",
    "upper_bound",
    "price_to_median_ratio",
    "status",
    "link",
]

outlier_df = outlier_df[[column for column in outlier_columns if column in outlier_df.columns]]
outlier_df = outlier_df.sort_values(["keyword", "outlier_type", "price_numeric"])

print(f"outlier rows: {len(outlier_df):,}")
outlier_df.head(50)


outlier rows: 5,300


,keyword,platform,pid,name,price_numeric,outlier_type,lower_bound,iqr_lower_bound,median_ratio_lower_bound,q1,median_price,q3,upper_bound,price_to_median_ratio,status,link
5975,5090,중고나라,228457956,5090 3개 일괄 판매합니다,16500000,high,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,2.844828e+00,거래완료,https://web.joongna.com/product/228457956
5851,5090,중고나라,228636935,RTX 5090 슈프림 -> 5090 아스트랄 화이트 교환 하실분,1,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.724138e-07,판매중,https://web.joongna.com/product/228636935
5881,5090,중고나라,223733140,사기 5070 5080 4090 5090 9950 9800 p41 p51 990 9910,112,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.931034e-05,판매중,https://web.joongna.com/product/223733140
5712,5090,번개장터,397111572,RTX5090 대량 매입합니다! 모델따지지않고 A/S기간 따지지않고 작동,550,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,9.482759e-05,판매중,https://m.bunjang.co.kr/products/397111572
5713,5090,번개장터,368439995,RTX5090삽니다! 삽니다 !,550,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,9.482759e-05,판매중,https://m.bunjang.co.kr/products/368439995
5912,5090,중고나라,228724580,ROG ASTRAL 5090,550,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,9.482759e-05,거래완료,https://web.joongna.com/product/228724580
5941,5090,중고나라,228509681,9800X3D 5090 엔드급 초고사양 하이엔드 컴퓨터 본체 PC,1004,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.731034e-04,거래완료,https://web.joongna.com/product/228509681
5724,5090,번개장터,408316423,"9800x3d, B850m wifi, rtx 5080 화이트 완본체",1111,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.915517e-04,예약중,https://m.bunjang.co.kr/products/408316423
5760,5090,번개장터,407704198,"9800x3d, b850m wifi, rtx 5080 완본체",1111,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.915517e-04,예약중,https://m.bunjang.co.kr/products/407704198
5770,5090,번개장터,407619560,"9800x3d, B850m wifi 7, rtx 5080 화이트 완본체",1111,low,1160000.0,0.0,1160000.0,3680000.0,5800000.0,7825000.0,14042500.0,1.915517e-04,예약중,https://m.bunjang.co.kr/products/407619560


In [19]:
outlier_summary_df = (
    outlier_df.groupby(["keyword", "outlier_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in ["low", "high"]:
    if column not in outlier_summary_df.columns:
        outlier_summary_df[column] = 0

outlier_summary_df["total_outlier_count"] = outlier_summary_df["low"] + outlier_summary_df["high"]
outlier_summary_df = outlier_summary_df.merge(
    keyword_stats_df[[
        "keyword",
        "item_count",
        "min_price",
        "median_price",
        "max_price",
        "lower_bound",
        "upper_bound",
    ]],
    on="keyword",
    how="left",
)
outlier_summary_df["outlier_rate"] = outlier_summary_df["total_outlier_count"] / outlier_summary_df["item_count"]
outlier_summary_df = outlier_summary_df.sort_values("total_outlier_count", ascending=False)

outlier_summary_df.head(30)


,keyword,high,low,total_outlier_count,item_count,min_price,median_price,max_price,lower_bound,upper_bound,outlier_rate
4,갤럭시,942,1458,2400,10414,500,159000.0,999999999,31800.0,7.016250e+05,0.230459
16,스텔라이브,202,370,572,1868,1,28000.0,999999999,5600.0,1.600000e+05,0.306210
14,세이코,241,149,390,1986,168,250000.0,12250000,50000.0,1.090000e+06,0.196375
19,핫토이,109,260,369,1812,100,220000.0,999999999,44000.0,7.073334e+05,0.203642
8,만년필,121,158,279,996,100,150000.0,99999999,30000.0,1.057500e+06,0.280120
2,ps5,185,30,215,1018,1,45000.0,90105000,9000.0,2.645625e+05,0.211198
6,골드바,105,109,214,526,250,820000.0,999999999,164000.0,1.995000e+06,0.406844
7,던스,72,71,143,1243,1,60000.0,4444444,12000.0,2.100000e+05,0.115044
5,게이밍 노트북,54,87,141,885,500,950000.0,9500000,190000.0,3.150000e+06,0.159322
11,메탈빌드,20,60,80,651,1230,327000.0,99999999,65400.0,8.837500e+05,0.122888


In [20]:
keyword_stats_path = OUTPUT_DIR / "keyword_price_stats.csv"
outlier_detail_path = OUTPUT_DIR / "keyword_price_outliers.csv"
outlier_summary_path = OUTPUT_DIR / "keyword_price_outlier_summary.csv"

keyword_stats_df.to_csv(keyword_stats_path, index=False, encoding="utf-8-sig")
outlier_df.to_csv(outlier_detail_path, index=False, encoding="utf-8-sig")
outlier_summary_df.to_csv(outlier_summary_path, index=False, encoding="utf-8-sig")

print(f"saved keyword stats: {keyword_stats_path}")
print(f"saved outlier detail: {outlier_detail_path}")
print(f"saved outlier summary: {outlier_summary_path}")


saved keyword stats: C:\project\kdtproject\kdtproject\code\backend\src\main\python\analysis\results\price_outliers\keyword_price_stats.csv
saved outlier detail: C:\project\kdtproject\kdtproject\code\backend\src\main\python\analysis\results\price_outliers\keyword_price_outliers.csv
saved outlier summary: C:\project\kdtproject\kdtproject\code\backend\src\main\python\analysis\results\price_outliers\keyword_price_outlier_summary.csv


In [21]:
overall_outlier_overview = {
    "source_file": csv_path.name,
    "valid_price_rows": len(working_df),
    "keyword_count": working_df["keyword"].nunique(),
    "outlier_rows": len(outlier_df),
    "outlier_rate": len(outlier_df) / len(working_df) if len(working_df) else 0,
    "low_outlier_rows": int((outlier_df["outlier_type"] == "low").sum()) if len(outlier_df) else 0,
    "high_outlier_rows": int((outlier_df["outlier_type"] == "high").sum()) if len(outlier_df) else 0,
}

overall_outlier_overview_df = pd.DataFrame([overall_outlier_overview])

keyword_outlier_overview_df = outlier_summary_df[[
    "keyword",
    "item_count",
    "low",
    "high",
    "total_outlier_count",
    "outlier_rate",
    "min_price",
    "median_price",
    "max_price",
    "lower_bound",
    "upper_bound",
]].sort_values(["total_outlier_count", "outlier_rate"], ascending=[False, False])

keyword_outlier_overview_path = OUTPUT_DIR / "keyword_price_outlier_overview.csv"
overall_outlier_overview_path = OUTPUT_DIR / "overall_price_outlier_overview.csv"
keyword_outlier_overview_df.to_csv(keyword_outlier_overview_path, index=False, encoding="utf-8-sig")
overall_outlier_overview_df.to_csv(overall_outlier_overview_path, index=False, encoding="utf-8-sig")

print("전체 요약")
display(overall_outlier_overview_df)

print("keyword별 이상치 많은 순")
display(keyword_outlier_overview_df.head(30))

print(f"saved keyword overview: {keyword_outlier_overview_path}")
print(f"saved overall overview: {overall_outlier_overview_path}")

전체 요약


,source_file,valid_price_rows,keyword_count,outlier_rows,outlier_rate,low_outlier_rows,high_outlier_rows
0,통합조회_전체_no_filter_20260522_1106.csv,24988,20,5300,0.212102,3059,2241


keyword별 이상치 많은 순


,keyword,item_count,low,high,total_outlier_count,outlier_rate,min_price,median_price,max_price,lower_bound,upper_bound
4,갤럭시,10414,1458,942,2400,0.230459,500,159000.0,999999999,31800.0,7.016250e+05
16,스텔라이브,1868,370,202,572,0.306210,1,28000.0,999999999,5600.0,1.600000e+05
14,세이코,1986,149,241,390,0.196375,168,250000.0,12250000,50000.0,1.090000e+06
19,핫토이,1812,260,109,369,0.203642,100,220000.0,999999999,44000.0,7.073334e+05
8,만년필,996,158,121,279,0.280120,100,150000.0,99999999,30000.0,1.057500e+06
2,ps5,1018,30,185,215,0.211198,1,45000.0,90105000,9000.0,2.645625e+05
6,골드바,526,109,105,214,0.406844,250,820000.0,999999999,164000.0,1.995000e+06
7,던스,1243,71,72,143,0.115044,1,60000.0,4444444,12000.0,2.100000e+05
5,게이밍 노트북,885,87,54,141,0.159322,500,950000.0,9500000,190000.0,3.150000e+06
11,메탈빌드,651,60,20,80,0.122888,1230,327000.0,99999999,65400.0,8.837500e+05


saved keyword overview: C:\project\kdtproject\kdtproject\code\backend\src\main\python\analysis\results\price_outliers\keyword_price_outlier_overview.csv
saved overall overview: C:\project\kdtproject\kdtproject\code\backend\src\main\python\analysis\results\price_outliers\overall_price_outlier_overview.csv


In [22]:
# 이상치 제거 후 keyword별 가격 요약 DF
outlier_keys = outlier_df[["keyword", "platform", "pid", "price_numeric", "outlier_type"]].copy()
outlier_keys["is_outlier"] = True

clean_price_df = working_df.merge(
    outlier_keys[["keyword", "platform", "pid", "price_numeric", "is_outlier"]],
    on=["keyword", "platform", "pid", "price_numeric"],
    how="left",
)
clean_price_df["is_outlier"] = clean_price_df["is_outlier"].fillna(False).astype(bool)
clean_price_df = clean_price_df.loc[~clean_price_df["is_outlier"]].copy()

clean_keyword_price_summary_df = (
    clean_price_df.groupby("keyword")["price_numeric"]
    .agg(
        clean_item_count="count",
        clean_min_price="min",
        clean_max_price="max",
        clean_average_price="mean",
        clean_median_price="median",
    )
    .reset_index()
)

original_counts_df = working_df.groupby("keyword").size().reset_index(name="original_item_count")
outlier_counts_df = outlier_df.groupby("keyword").size().reset_index(name="removed_outlier_count")

clean_keyword_price_summary_df = original_counts_df.merge(
    outlier_counts_df,
    on="keyword",
    how="left",
).merge(
    clean_keyword_price_summary_df,
    on="keyword",
    how="left",
)
clean_keyword_price_summary_df["removed_outlier_count"] = clean_keyword_price_summary_df["removed_outlier_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["clean_item_count"] = clean_keyword_price_summary_df["clean_item_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["removed_outlier_rate"] = clean_keyword_price_summary_df["removed_outlier_count"] / clean_keyword_price_summary_df["original_item_count"]

clean_keyword_price_summary_df = clean_keyword_price_summary_df[[
    "keyword",
    "original_item_count",
    "removed_outlier_count",
    "removed_outlier_rate",
    "clean_item_count",
    "clean_min_price",
    "clean_max_price",
    "clean_average_price",
    "clean_median_price",
]].sort_values("keyword")

clean_keyword_price_summary_df

KeyError: "None of [Index([-1, -2, -2, -1, -1, -2, -1, -1, -1, -1,\n       ...\n       -1, -1, -1, -1, -1, -1, -1, -1, -2, -1],\n      dtype='object', length=24988)] are in the [columns]"